# 🔧 Système d'Analyse de Soudure par Intelligence Artificielle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbambi/Ai-test/blob/main/Weld_Analysis_Colab.ipynb)

## 📋 Fonctionnalités

- 🎯 **Segmentation automatique** des zones de soudure (U-Net)
- 📏 **Mesure de la largeur** du cordon de soudure
- 🔗 **Analyse de la continuité** (détection des interruptions)
- 🎨 **Évaluation de l'homogénéité** (CNN)
- ⭐ **Score de qualité** (notation A-F)
- 🎓 **Entraînement personnalisé** avec vos propres données

---

## 🚀 1. Installation et Configuration

Exécutez cette cellule pour installer toutes les dépendances nécessaires.

In [ ]:
#@title ⚙️ Installation des dépendances {display-mode: "form"}
#@markdown Cliquez sur le bouton ▶️ pour installer les bibliothèques nécessaires.

import subprocess
import sys

def install_packages():
    """Installe les packages nécessaires."""
    packages = [
        'torch',
        'torchvision', 
        'opencv-python',
        'scikit-image',
        'albumentations',
        'tqdm',
        'ipywidgets',
        'gdown'
    ]
    
    print("📦 Installation des dépendances...")
    for package in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    
    print("✅ Installation terminée!")

install_packages()

# Activer les widgets
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
#@title 📚 Importation des bibliothèques {display-mode: "form"}

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import base64
from tqdm.notebook import tqdm
from scipy import ndimage
from skimage import morphology
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files
import gdown
import zipfile
import warnings
warnings.filterwarnings('ignore')

# Vérifier GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Appareil utilisé: {device.upper()}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print("✅ Bibliothèques importées!")

---
## 🧠 2. Définition des Modèles

Cette section contient les architectures des réseaux de neurones.

In [ ]:
#@title 🏗️ Architecture U-Net pour la Segmentation {display-mode: "form"}
#@markdown U-Net est un réseau de neurones convolutif pour la segmentation d'images.

class DoubleConv(nn.Module):
    """Bloc de double convolution (Conv -> BN -> ReLU) x 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Bloc de sous-échantillonnage (MaxPool -> DoubleConv)"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Bloc de sur-échantillonnage avec skip connection"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """
    Architecture U-Net complète pour la segmentation de soudures.
    
    Paramètres:
    - n_channels: Nombre de canaux d'entrée (1 pour niveaux de gris)
    - n_classes: Nombre de classes de sortie (1 pour binaire)
    - bilinear: Utiliser l'interpolation bilinéaire pour le upsampling
    """
    def __init__(self, n_channels=1, n_classes=1, bilinear=True):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        factor = 2 if bilinear else 1

        # Encodeur
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        
        # Décodeur
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

    def predict(self, x, threshold=0.5):
        with torch.no_grad():
            logits = self.forward(x)
            probs = torch.sigmoid(logits)
            return (probs > threshold).float()

print("✅ Modèle U-Net défini!")
model_unet = UNet(n_channels=1, n_classes=1).to(device)
total_params = sum(p.numel() for p in model_unet.parameters())
print(f"   Paramètres: {total_params:,}")

In [ ]:
#@title 🏗️ CNN pour la Classification d'Homogénéité {display-mode: "form"}
#@markdown Classifie les zones de soudure selon leur homogénéité.

class HomogeneityClassifier(nn.Module):
    """
    CNN pour classifier l'homogénéité des soudures.
    
    Classes:
    - 0: Non homogène (défauts majeurs)
    - 1: Partiellement homogène (défauts mineurs)
    - 2: Homogène (qualité acceptable)
    """
    def __init__(self, n_channels=1, n_classes=3, dropout=0.3):
        super().__init__()
        
        self.features = nn.Sequential(
            # Bloc 1
            nn.Conv2d(n_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            # Bloc 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout * 0.5),
            
            # Bloc 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)
    
    def predict(self, x):
        with torch.no_grad():
            logits = self.forward(x)
            probs = F.softmax(logits, dim=1)
            return probs.argmax(dim=1), probs

print("✅ Classificateur d'homogénéité défini!")
model_classifier = HomogeneityClassifier().to(device)
total_params = sum(p.numel() for p in model_classifier.parameters())
print(f"   Paramètres: {total_params:,}")

---
## 🔧 3. Outils d'Analyse

Fonctions pour analyser les zones de soudure.

In [ ]:
#@title 📐 Analyseur de Soudure {display-mode: "form"}
#@markdown Outils pour mesurer la largeur, continuité et homogénéité.

class WeldAnalyzer:
    """
    Analyseur complet de zones de soudure.
    
    Mesure:
    - Largeur du cordon
    - Continuité (détection des interruptions)
    - Homogénéité de la texture
    - Score de qualité global
    """
    
    def __init__(self, pixels_per_mm=10.0):
        self.pixels_per_mm = pixels_per_mm
    
    def analyze(self, image, mask):
        """Analyse complète d'une zone de soudure."""
        mask_binary = (mask > 127).astype(np.uint8)
        
        # Mesure de largeur
        width_stats = self._measure_width(mask_binary)
        
        # Analyse de continuité
        continuity = self._analyze_continuity(mask_binary)
        
        # Analyse d'homogénéité
        homogeneity = self._analyze_homogeneity(image, mask_binary)
        
        # Score global
        quality_score = self._compute_quality_score(
            width_stats, continuity, homogeneity
        )
        
        return {
            'width': width_stats,
            'continuity': continuity,
            'homogeneity': homogeneity,
            'quality_score': quality_score,
            'grade': self._get_grade(quality_score),
            'is_acceptable': quality_score >= 0.6
        }
    
    def _measure_width(self, mask):
        """Mesure la largeur du cordon de soudure."""
        if np.sum(mask) == 0:
            return {'mean': 0, 'std': 0, 'min': 0, 'max': 0}
        
        # Distance transform
        dist = ndimage.distance_transform_edt(mask)
        skeleton = morphology.skeletonize(mask > 0)
        
        # Largeur = 2 * distance au bord
        widths = dist[skeleton] * 2
        
        if len(widths) == 0:
            return {'mean': 0, 'std': 0, 'min': 0, 'max': 0}
        
        return {
            'mean': np.mean(widths),
            'std': np.std(widths),
            'min': np.min(widths),
            'max': np.max(widths),
            'mean_mm': np.mean(widths) / self.pixels_per_mm
        }
    
    def _analyze_continuity(self, mask):
        """Analyse la continuité du cordon."""
        labeled, n_labels = ndimage.label(mask)
        
        if n_labels == 0:
            return {'score': 0, 'n_gaps': 0}
        
        if n_labels == 1:
            return {'score': 1.0, 'n_gaps': 0}
        
        # Taille de chaque composante
        sizes = [np.sum(labeled == i) for i in range(1, n_labels + 1)]
        main_size = max(sizes)
        total_size = sum(sizes)
        
        return {
            'score': main_size / total_size,
            'n_gaps': n_labels - 1,
            'n_components': n_labels
        }
    
    def _analyze_homogeneity(self, image, mask):
        """Analyse l'homogénéité de la texture."""
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image
        
        mask_bool = mask > 0
        if np.sum(mask_bool) == 0:
            return {'score': 0, 'texture_uniformity': 0, 'defect_ratio': 1}
        
        weld_pixels = gray[mask_bool]
        
        # Uniformité de texture (inverse de la variance normalisée)
        mean_val = np.mean(weld_pixels)
        std_val = np.std(weld_pixels)
        cv = std_val / (mean_val + 1e-6)
        texture_uniformity = 1.0 / (1.0 + cv * 2)
        
        # Détection de défauts (pixels anormaux)
        z_scores = np.abs(weld_pixels - mean_val) / (std_val + 1e-6)
        defect_ratio = np.mean(z_scores > 2.5)
        
        score = 0.6 * texture_uniformity + 0.4 * (1 - defect_ratio)
        
        return {
            'score': float(score),
            'texture_uniformity': float(texture_uniformity),
            'defect_ratio': float(defect_ratio)
        }
    
    def _compute_quality_score(self, width, continuity, homogeneity):
        """Calcule le score de qualité global."""
        # Score de largeur (pénaliser la variabilité)
        width_cv = width['std'] / (width['mean'] + 1e-6)
        width_score = 1.0 / (1.0 + width_cv)
        
        # Pondération
        score = (
            0.2 * width_score +
            0.4 * continuity['score'] +
            0.4 * homogeneity['score']
        )
        
        return float(np.clip(score, 0, 1))
    
    def _get_grade(self, score):
        """Convertit le score en note."""
        if score >= 0.9: return 'A'
        if score >= 0.8: return 'B'
        if score >= 0.7: return 'C'
        if score >= 0.6: return 'D'
        return 'F'

# Instance globale
analyzer = WeldAnalyzer(pixels_per_mm=10.0)
print("✅ Analyseur de soudure créé!")

---
## 📊 4. Dataset et Entraînement

In [ ]:
#@title 📁 Dataset Synthétique pour l'Entraînement {display-mode: "form"}
#@markdown Génère des données de soudure synthétiques pour l'entraînement.

class SyntheticWeldDataset(Dataset):
    """
    Dataset synthétique de soudures pour l'entraînement.
    Génère des images avec des cordons de soudure simulés.
    """
    
    def __init__(self, n_samples=1000, image_size=(256, 256), augment=True):
        self.n_samples = n_samples
        self.image_size = image_size
        self.augment = augment
        np.random.seed(42)
        self.params = [self._generate_params() for _ in range(n_samples)]
    
    def _generate_params(self):
        h, w = self.image_size
        return {
            'weld_y': np.random.randint(h // 4, 3 * h // 4),
            'weld_width': np.random.randint(15, 50),
            'weld_length': np.random.randint(w // 2, w - 20),
            'weld_start': np.random.randint(10, w // 4),
            'noise_level': np.random.uniform(10, 25),
            'has_gap': np.random.random() < 0.15,
            'gap_x': np.random.randint(w // 3, 2 * w // 3),
            'gap_width': np.random.randint(5, 15),
            'weld_intensity': np.random.randint(160, 200),
            'bg_intensity': np.random.randint(80, 120)
        }
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        params = self.params[idx]
        h, w = self.image_size
        
        # Fond avec bruit
        image = np.random.normal(
            params['bg_intensity'], 15, (h, w)
        ).astype(np.float32)
        
        # Masque
        mask = np.zeros((h, w), dtype=np.float32)
        
        # Coordonnées de la soudure
        y1 = max(0, params['weld_y'] - params['weld_width'] // 2)
        y2 = min(h, params['weld_y'] + params['weld_width'] // 2)
        x1 = params['weld_start']
        x2 = min(w, x1 + params['weld_length'])
        
        # Dessiner la soudure
        mask[y1:y2, x1:x2] = 1.0
        
        # Ajouter une interruption si demandé
        if params['has_gap']:
            gx1 = params['gap_x']
            gx2 = min(w, gx1 + params['gap_width'])
            mask[y1:y2, gx1:gx2] = 0.0
        
        # Texture de la soudure
        weld_texture = np.random.normal(
            params['weld_intensity'], 10, (h, w)
        ).astype(np.float32)
        image = np.where(mask > 0.5, weld_texture, image)
        
        # Bruit
        noise = np.random.normal(0, params['noise_level'], (h, w))
        image = image + noise
        
        # Normaliser
        image = np.clip(image, 0, 255) / 255.0
        
        # Augmentation simple
        if self.augment and np.random.random() < 0.5:
            image = np.fliplr(image).copy()
            mask = np.fliplr(mask).copy()
        
        # Convertir en tenseurs
        image_tensor = torch.from_numpy(image).unsqueeze(0).float()
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image_tensor, mask_tensor

print("✅ Dataset synthétique défini!")

In [ ]:
#@title 🎓 Fonction d'Entraînement {display-mode: "form"}
#@markdown Entraîne le modèle U-Net avec les paramètres spécifiés.

class DiceLoss(nn.Module):
    """Dice Loss pour la segmentation."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice


class CombinedLoss(nn.Module):
    """BCE + Dice Loss combinées."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    
    def forward(self, pred, target):
        return 0.5 * self.bce(pred, target) + 0.5 * self.dice(pred, target)


def train_model(model, train_loader, val_loader, epochs=50, lr=1e-4, 
                progress_callback=None):
    """
    Entraîne le modèle de segmentation.
    
    Args:
        model: Modèle U-Net
        train_loader: DataLoader d'entraînement
        val_loader: DataLoader de validation
        epochs: Nombre d'epochs
        lr: Learning rate
        progress_callback: Fonction de callback pour la progression
    
    Returns:
        Historique d'entraînement
    """
    criterion = CombinedLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    
    history = {'train_loss': [], 'val_loss': [], 'val_iou': []}
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Entraînement
        model.train()
        train_loss = 0
        
        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0
        val_iou = 0
        
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                
                # IoU
                preds = (torch.sigmoid(outputs) > 0.5).float()
                intersection = (preds * masks).sum()
                union = preds.sum() + masks.sum() - intersection
                val_iou += (intersection / (union + 1e-8)).item()
        
        val_loss /= len(val_loader)
        val_iou /= len(val_loader)
        
        # Sauvegarder historique
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_iou'].append(val_iou)
        
        # Scheduler
        scheduler.step(val_loss)
        
        # Sauvegarder meilleur modèle
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
        
        # Callback
        if progress_callback:
            progress_callback(epoch + 1, epochs, train_loss, val_loss, val_iou)
    
    # Charger le meilleur modèle
    model.load_state_dict(torch.load('best_model.pth'))
    
    return history

print("✅ Fonction d'entraînement définie!")

---
## 🖥️ 5. Interface Interactive

Interface graphique pour utiliser le système.

In [ ]:
#@title 🎨 Fonctions d'Affichage {display-mode: "form"}

def preprocess_image(image, target_size=(256, 256)):
    """Prétraite une image pour le modèle."""
    # Convertir en niveaux de gris si nécessaire
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image
    
    # Redimensionner
    resized = cv2.resize(gray, target_size)
    
    # Améliorer le contraste
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(resized)
    
    return enhanced


def visualize_results(image, mask, analysis, figsize=(15, 5)):
    """Visualise les résultats d'analyse."""
    fig, axes = plt.subplots(1, 4, figsize=figsize)
    
    # Image originale
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Image Originale')
    axes[0].axis('off')
    
    # Masque de segmentation
    axes[1].imshow(mask, cmap='hot')
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    
    # Overlay
    overlay = np.stack([image] * 3, axis=-1).astype(np.float32) / 255
    mask_colored = np.zeros_like(overlay)
    mask_colored[:, :, 1] = (mask > 127).astype(np.float32)  # Vert
    combined = overlay * 0.7 + mask_colored * 0.3
    axes[2].imshow(combined)
    axes[2].set_title('Superposition')
    axes[2].axis('off')
    
    # Métriques
    axes[3].axis('off')
    metrics_text = f"""
    📊 RÉSULTATS D'ANALYSE
    ═══════════════════════════
    
    Note: {analysis['grade']}
    Score: {analysis['quality_score']:.1%}
    
    📏 Largeur moyenne: {analysis['width']['mean']:.1f} px
       ({analysis['width'].get('mean_mm', 0):.2f} mm)
    
    🔗 Continuité: {analysis['continuity']['score']:.1%}
       Interruptions: {analysis['continuity']['n_gaps']}
    
    🎨 Homogénéité: {analysis['homogeneity']['score']:.1%}
       Défauts: {analysis['homogeneity']['defect_ratio']:.1%}
    
    {'✅ ACCEPTABLE' if analysis['is_acceptable'] else '❌ NON CONFORME'}
    """
    axes[3].text(0.1, 0.5, metrics_text, transform=axes[3].transAxes,
                 fontsize=11, verticalalignment='center',
                 fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()


def plot_training_history(history):
    """Affiche l'historique d'entraînement."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Évolution de la Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # IoU
    axes[1].plot(history['val_iou'], color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('IoU')
    axes[1].set_title('IoU de Validation')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✅ Fonctions d'affichage définies!")

In [ ]:
#@title 🖥️ Interface Interactive Principale {display-mode: "form"}
#@markdown Exécutez cette cellule pour afficher l'interface interactive.

# Variables globales pour stocker l'état
uploaded_image = None
current_mask = None
current_analysis = None

# Widgets
output_area = widgets.Output()

# Titre
title = widgets.HTML(
    value="""
    <h1 style='text-align: center; color: #2E86AB;'>
        🔧 Système d'Analyse de Soudure par IA
    </h1>
    <hr>
    """
)

# Sélection du mode
mode_dropdown = widgets.Dropdown(
    options=[
        ('📷 Analyser une Image', 'analyze'),
        ('🎓 Entraîner le Modèle', 'train'),
        ('📊 Démonstration', 'demo')
    ],
    value='demo',
    description='Mode:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='300px')
)

# Bouton d'upload
upload_button = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📁 Charger Image',
    layout=widgets.Layout(width='200px')
)

# Bouton d'exécution
run_button = widgets.Button(
    description='▶️ Exécuter',
    button_style='success',
    layout=widgets.Layout(width='150px')
)

# Paramètres d'entraînement
epochs_slider = widgets.IntSlider(
    value=30,
    min=5,
    max=100,
    step=5,
    description='Epochs:',
    style={'description_width': '80px'}
)

samples_slider = widgets.IntSlider(
    value=500,
    min=100,
    max=2000,
    step=100,
    description='Samples:',
    style={'description_width': '80px'}
)

batch_slider = widgets.IntSlider(
    value=8,
    min=2,
    max=32,
    step=2,
    description='Batch:',
    style={'description_width': '80px'}
)

lr_dropdown = widgets.Dropdown(
    options=[
        ('0.001', 1e-3),
        ('0.0001', 1e-4),
        ('0.00001', 1e-5)
    ],
    value=1e-4,
    description='LR:',
    style={'description_width': '80px'}
)

# Progress bar
progress_bar = widgets.FloatProgress(
    value=0,
    min=0,
    max=100,
    description='Progression:',
    bar_style='info',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='100%', visibility='hidden')
)

status_label = widgets.HTML(value="")

# Panneaux de paramètres
train_params = widgets.VBox([
    widgets.HTML("<b>⚙️ Paramètres d'entraînement:</b>"),
    epochs_slider,
    samples_slider,
    batch_slider,
    lr_dropdown
], layout=widgets.Layout(display='none'))

analyze_params = widgets.VBox([
    widgets.HTML("<b>📷 Charger une image de soudure:</b>"),
    upload_button
], layout=widgets.Layout(display='none'))


def on_mode_change(change):
    """Change l'affichage selon le mode sélectionné."""
    mode = change['new']
    
    if mode == 'train':
        train_params.layout.display = 'block'
        analyze_params.layout.display = 'none'
    elif mode == 'analyze':
        train_params.layout.display = 'none'
        analyze_params.layout.display = 'block'
    else:
        train_params.layout.display = 'none'
        analyze_params.layout.display = 'none'

mode_dropdown.observe(on_mode_change, names='value')


def on_upload_change(change):
    """Traite l'image uploadée."""
    global uploaded_image
    
    if upload_button.value:
        file_info = list(upload_button.value.values())[0]
        content = file_info['content']
        
        # Convertir en image
        image = Image.open(BytesIO(content))
        uploaded_image = np.array(image)
        
        with output_area:
            clear_output(wait=True)
            print(f"✅ Image chargée: {uploaded_image.shape}")
            plt.figure(figsize=(6, 6))
            plt.imshow(uploaded_image, cmap='gray' if len(uploaded_image.shape) == 2 else None)
            plt.title('Image Chargée')
            plt.axis('off')
            plt.show()

upload_button.observe(on_upload_change, names='value')


def run_demo():
    """Exécute une démonstration avec des données synthétiques."""
    global current_mask, current_analysis
    
    with output_area:
        clear_output(wait=True)
        print("🎬 Démonstration en cours...\n")
        
        # Générer une image synthétique
        dataset = SyntheticWeldDataset(n_samples=1, augment=False)
        image_tensor, mask_tensor = dataset[0]
        
        # Convertir en numpy
        image = (image_tensor[0].numpy() * 255).astype(np.uint8)
        true_mask = (mask_tensor[0].numpy() * 255).astype(np.uint8)
        
        # Prédiction avec le modèle
        model_unet.eval()
        with torch.no_grad():
            pred = model_unet(image_tensor.unsqueeze(0).to(device))
            pred_mask = (torch.sigmoid(pred) > 0.5).float()
            pred_mask = (pred_mask[0, 0].cpu().numpy() * 255).astype(np.uint8)
        
        # Analyse
        current_mask = true_mask  # Utiliser le vrai masque pour la démo
        current_analysis = analyzer.analyze(image, true_mask)
        
        # Visualiser
        visualize_results(image, true_mask, current_analysis)


def run_analysis():
    """Analyse l'image uploadée."""
    global uploaded_image, current_mask, current_analysis
    
    if uploaded_image is None:
        with output_area:
            clear_output(wait=True)
            print("❌ Veuillez d'abord charger une image!")
        return
    
    with output_area:
        clear_output(wait=True)
        print("🔍 Analyse en cours...\n")
        
        # Prétraiter
        processed = preprocess_image(uploaded_image)
        
        # Convertir en tenseur
        tensor = torch.from_numpy(processed).float().unsqueeze(0).unsqueeze(0) / 255.0
        tensor = tensor.to(device)
        
        # Prédiction
        model_unet.eval()
        with torch.no_grad():
            output = model_unet(tensor)
            mask = (torch.sigmoid(output) > 0.5).float()
            mask = (mask[0, 0].cpu().numpy() * 255).astype(np.uint8)
        
        # Redimensionner le masque à la taille originale si nécessaire
        h, w = uploaded_image.shape[:2]
        mask = cv2.resize(mask, (w, h))
        
        # Analyse
        gray = cv2.cvtColor(uploaded_image, cv2.COLOR_RGB2GRAY) if len(uploaded_image.shape) == 3 else uploaded_image
        current_mask = mask
        current_analysis = analyzer.analyze(gray, mask)
        
        # Visualiser
        visualize_results(gray, mask, current_analysis)


def run_training():
    """Entraîne le modèle avec les paramètres sélectionnés."""
    global model_unet
    
    epochs = epochs_slider.value
    n_samples = samples_slider.value
    batch_size = batch_slider.value
    lr = lr_dropdown.value
    
    with output_area:
        clear_output(wait=True)
        print(f"🎓 Démarrage de l'entraînement...")
        print(f"   Epochs: {epochs}")
        print(f"   Samples: {n_samples}")
        print(f"   Batch size: {batch_size}")
        print(f"   Learning rate: {lr}")
        print()
        
        # Créer le dataset
        train_dataset = SyntheticWeldDataset(n_samples=int(n_samples * 0.8))
        val_dataset = SyntheticWeldDataset(n_samples=int(n_samples * 0.2), augment=False)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
        
        print(f"📊 Dataset créé: {len(train_dataset)} train, {len(val_dataset)} val\n")
        
        # Réinitialiser le modèle
        model_unet = UNet(n_channels=1, n_classes=1).to(device)
        
        # Progress callback
        def update_progress(epoch, total, train_loss, val_loss, val_iou):
            progress_bar.value = (epoch / total) * 100
            status_label.value = f"<b>Epoch {epoch}/{total}</b> | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | IoU: {val_iou:.4f}"
        
        progress_bar.layout.visibility = 'visible'
        
        # Entraîner
        history = train_model(
            model_unet, 
            train_loader, 
            val_loader, 
            epochs=epochs, 
            lr=lr,
            progress_callback=update_progress
        )
        
        progress_bar.layout.visibility = 'hidden'
        
        print("\n✅ Entraînement terminé!")
        print(f"   Meilleur IoU: {max(history['val_iou']):.4f}")
        print(f"   Meilleure Loss: {min(history['val_loss']):.4f}")
        
        # Afficher l'historique
        plot_training_history(history)


def on_run_clicked(b):
    """Handler du bouton Exécuter."""
    mode = mode_dropdown.value
    
    if mode == 'demo':
        run_demo()
    elif mode == 'analyze':
        run_analysis()
    elif mode == 'train':
        run_training()

run_button.on_click(on_run_clicked)


# Bouton télécharger le modèle
download_button = widgets.Button(
    description='💾 Télécharger Modèle',
    button_style='info',
    layout=widgets.Layout(width='180px')
)

def on_download_clicked(b):
    torch.save(model_unet.state_dict(), 'weld_segmentation_model.pth')
    files.download('weld_segmentation_model.pth')

download_button.on_click(on_download_clicked)


# Assembler l'interface
controls = widgets.HBox([
    mode_dropdown,
    run_button,
    download_button
], layout=widgets.Layout(justify_content='flex-start', gap='20px'))

params_area = widgets.VBox([train_params, analyze_params])

interface = widgets.VBox([
    title,
    controls,
    params_area,
    progress_bar,
    status_label,
    widgets.HTML("<hr>"),
    output_area
], layout=widgets.Layout(padding='20px'))

display(interface)

# Message initial
with output_area:
    print("👋 Bienvenue dans le système d'analyse de soudure!")
    print()
    print("📋 Modes disponibles:")
    print("   • Démonstration: Teste le système avec des données synthétiques")
    print("   • Analyser une Image: Chargez votre propre image de soudure")
    print("   • Entraîner le Modèle: Entraînez le réseau avec des données synthétiques")
    print()
    print("🚀 Sélectionnez un mode et cliquez sur 'Exécuter' pour commencer!")

---
## 📥 6. Téléchargement de Dataset (Optionnel)

Téléchargez un dataset de soudures réelles pour un entraînement avancé.

In [ ]:
#@title 📥 Télécharger un Dataset de Soudures {display-mode: "form"}
#@markdown Sélectionnez et téléchargez un dataset public de soudures.

dataset_choice = "GDXray (Welds)" #@param ["GDXray (Welds)", "Synthetic Only", "Custom Upload"]

def download_gdxray():
    """Télécharge le dataset GDXray (partie soudures)."""
    print("📥 Téléchargement du dataset GDXray (Welds)...")
    print("   Note: Le dataset complet est volumineux.")
    print("   Nous téléchargeons un sous-ensemble pour la démo.")
    print()
    
    # Créer le répertoire
    os.makedirs('dataset/images', exist_ok=True)
    os.makedirs('dataset/masks', exist_ok=True)
    
    # URL du dataset (à remplacer par un lien réel)
    # Pour la démo, on génère des données synthétiques
    print("⚠️ Lien direct non disponible. Génération de données synthétiques...")
    
    # Générer des données synthétiques
    dataset = SyntheticWeldDataset(n_samples=100, augment=False)
    
    for i, (img, mask) in enumerate(dataset):
        img_np = (img[0].numpy() * 255).astype(np.uint8)
        mask_np = (mask[0].numpy() * 255).astype(np.uint8)
        
        cv2.imwrite(f'dataset/images/weld_{i:04d}.png', img_np)
        cv2.imwrite(f'dataset/masks/weld_{i:04d}.png', mask_np)
    
    print(f"✅ Dataset créé: 100 images dans ./dataset/")
    print(f"   Images: ./dataset/images/")
    print(f"   Masques: ./dataset/masks/")

def upload_custom_dataset():
    """Permet l'upload d'un dataset personnalisé."""
    print("📁 Upload d'un dataset personnalisé")
    print()
    print("Structure attendue du fichier ZIP:")
    print("  dataset.zip")
    print("  ├── images/")
    print("  │   ├── img001.png")
    print("  │   └── ...")
    print("  └── masks/")
    print("      ├── img001.png")
    print("      └── ...")
    print()
    
    uploaded = files.upload()
    
    for filename, content in uploaded.items():
        if filename.endswith('.zip'):
            with open(filename, 'wb') as f:
                f.write(content)
            
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('dataset')
            
            print(f"✅ Dataset extrait dans ./dataset/")

if dataset_choice == "GDXray (Welds)":
    download_gdxray()
elif dataset_choice == "Custom Upload":
    upload_custom_dataset()
else:
    print("ℹ️ Mode synthétique: les données seront générées à la volée.")

---
## 🎯 7. Analyse Avancée

In [ ]:
#@title 📊 Analyse par Lot (Batch Analysis) {display-mode: "form"}
#@markdown Analysez plusieurs images d'un coup.

def batch_analysis():
    """Analyse un lot d'images."""
    print("📁 Sélectionnez plusieurs images de soudure:")
    uploaded = files.upload()
    
    if not uploaded:
        print("❌ Aucune image sélectionnée.")
        return
    
    results = []
    
    for filename, content in tqdm(uploaded.items(), desc="Analyse"):
        # Charger l'image
        image = Image.open(BytesIO(content))
        image_np = np.array(image)
        
        # Prétraiter
        processed = preprocess_image(image_np)
        
        # Prédire
        tensor = torch.from_numpy(processed).float().unsqueeze(0).unsqueeze(0) / 255.0
        tensor = tensor.to(device)
        
        model_unet.eval()
        with torch.no_grad():
            output = model_unet(tensor)
            mask = (torch.sigmoid(output) > 0.5).float()
            mask = (mask[0, 0].cpu().numpy() * 255).astype(np.uint8)
        
        # Redimensionner
        h, w = image_np.shape[:2]
        mask = cv2.resize(mask, (w, h))
        
        # Analyser
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY) if len(image_np.shape) == 3 else image_np
        analysis = analyzer.analyze(gray, mask)
        analysis['filename'] = filename
        results.append(analysis)
    
    # Afficher le résumé
    print("\n" + "="*60)
    print("📊 RÉSUMÉ DE L'ANALYSE PAR LOT")
    print("="*60)
    
    grades = [r['grade'] for r in results]
    scores = [r['quality_score'] for r in results]
    acceptable = sum(1 for r in results if r['is_acceptable'])
    
    print(f"\nImages analysées: {len(results)}")
    print(f"Score moyen: {np.mean(scores):.1%}")
    print(f"Acceptables: {acceptable}/{len(results)} ({acceptable/len(results):.1%})")
    print("\nDistribution des notes:")
    for grade in ['A', 'B', 'C', 'D', 'F']:
        count = grades.count(grade)
        bar = '█' * count
        print(f"  {grade}: {bar} {count}")
    
    print("\n" + "="*60)
    print("Détails par image:")
    print("="*60)
    for r in results:
        status = '✅' if r['is_acceptable'] else '❌'
        print(f"{status} {r['filename']}: {r['grade']} ({r['quality_score']:.1%})")
    
    return results

# Bouton pour lancer l'analyse par lot
batch_button = widgets.Button(
    description='📊 Analyser un Lot',
    button_style='warning',
    layout=widgets.Layout(width='180px')
)

batch_output = widgets.Output()

def on_batch_clicked(b):
    with batch_output:
        clear_output(wait=True)
        batch_analysis()

batch_button.on_click(on_batch_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>📊 Analyse par Lot</h3>"),
    widgets.HTML("<p>Analysez plusieurs images de soudure simultanément.</p>"),
    batch_button,
    batch_output
]))

---
## 💾 8. Sauvegarde et Chargement du Modèle

In [ ]:
#@title 💾 Sauvegarder / Charger le Modèle {display-mode: "form"}
#@markdown Gérez vos modèles entraînés.

action = "Sauvegarder" #@param ["Sauvegarder", "Charger", "Charger depuis Drive"]

if action == "Sauvegarder":
    # Sauvegarder
    model_path = 'weld_unet_model.pth'
    torch.save({
        'model_state_dict': model_unet.state_dict(),
        'model_config': {
            'n_channels': 1,
            'n_classes': 1,
            'bilinear': True
        }
    }, model_path)
    
    print(f"✅ Modèle sauvegardé: {model_path}")
    
    # Proposer le téléchargement
    download = True #@param {type:"boolean"}
    if download:
        files.download(model_path)

elif action == "Charger":
    print("📁 Sélectionnez un fichier de modèle (.pth):")
    uploaded = files.upload()
    
    for filename, content in uploaded.items():
        if filename.endswith('.pth'):
            # Sauvegarder temporairement
            with open(filename, 'wb') as f:
                f.write(content)
            
            # Charger
            checkpoint = torch.load(filename, map_location=device)
            
            if 'model_state_dict' in checkpoint:
                model_unet.load_state_dict(checkpoint['model_state_dict'])
            else:
                model_unet.load_state_dict(checkpoint)
            
            print(f"✅ Modèle chargé depuis {filename}")

elif action == "Charger depuis Drive":
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_path = '/content/drive/MyDrive/weld_model.pth' #@param {type:"string"}
    
    if os.path.exists(drive_path):
        checkpoint = torch.load(drive_path, map_location=device)
        if 'model_state_dict' in checkpoint:
            model_unet.load_state_dict(checkpoint['model_state_dict'])
        else:
            model_unet.load_state_dict(checkpoint)
        print(f"✅ Modèle chargé depuis Google Drive")
    else:
        print(f"❌ Fichier non trouvé: {drive_path}")

---
## 📚 Documentation

### 🎯 Objectif
Ce système permet d'identifier automatiquement les zones de soudure et d'évaluer leur qualité.

### 🧠 Modèles IA
- **U-Net**: Segmentation des zones de soudure
- **CNN**: Classification de l'homogénéité

### 📊 Métriques
- **Largeur**: Mesure du cordon en pixels/mm
- **Continuité**: Détection des interruptions
- **Homogénéité**: Uniformité de la texture
- **Score global**: Note A-F

### 📁 Datasets
- **GDXray**: Images industrielles publiques
- **Synthétique**: Données générées pour l'entraînement

---

**Développé pour le contrôle qualité industriel automatisé** 🔧